In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import json
import os
import numpy as np
from datetime import datetime
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

num_delete = int(0.05 * len(dataset))
indices = np.random.permutation(len(dataset))

delete_idx = indices[:num_delete]
retain_idx = indices[num_delete:]

retain_set = Subset(dataset, retain_idx)
delete_set = Subset(dataset, delete_idx)

retain_loader = DataLoader(retain_set, batch_size=64, shuffle=True)
delete_loader = DataLoader(delete_set, batch_size=64, shuffle=True)

print("✔ retain_loader and delete_loader ready")
print("Retain samples:", len(retain_set))
print("Delete samples:", len(delete_set))

✔ retain_loader and delete_loader ready
Retain samples: 57000
Delete samples: 3000


In [3]:
assert "retain_loader" in globals()
assert "delete_loader" in globals()

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(5408, 10)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
def train_with_checkpoint(
    model,
    loader,
    epochs=5,
    save_epoch=1,
    device="cpu"
):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    ckpt_path = f"early_checkpoint_epoch{save_epoch}.pt"

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x, y in loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        print(f"Epoch {epoch}: loss = {epoch_loss / len(loader):.4f}")

        if epoch == save_epoch:
            torch.save(model.state_dict(), ckpt_path)
            print(f"✔ Early checkpoint saved at epoch {epoch}")

    torch.save(model.state_dict(), "full_model.pt")
    print("✔ Full model saved")

    return {
        "total_epochs": epochs,
        "save_epoch": save_epoch,
        "checkpoint_path": ckpt_path,
        "epoch_indexing": "0-based"
    }

In [6]:
def first_epoch_reversal(
    model,
    retain_loader,
    checkpoint_path,
    epochs=3,
    device="cpu"
):
    model.load_state_dict(
        torch.load(checkpoint_path, map_location=device)
    )
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    start = time.time()

    model.train()
    for epoch in range(epochs):
        for x, y in retain_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

    unlearn_time = time.time() - start

    torch.save(model.state_dict(), "reversal_unlearned.pt")

    return model, unlearn_time

In [7]:
def evaluate(model, loader, device="cpu"):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            loss = criterion(outputs, y)

            loss_sum += loss.item()
            preds = outputs.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return {
        "accuracy": correct / total,
        "loss": loss_sum / len(loader)
    }


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleCNN()

train_meta = train_with_checkpoint(
    model,
    retain_loader,
    epochs=5,
    save_epoch=1,
    device=device
)

unlearned_model = SimpleCNN()

unlearned_model, unlearn_time = first_epoch_reversal(
    unlearned_model,
    retain_loader,
    checkpoint_path=train_meta["checkpoint_path"],
    epochs=3,
    device=device
)

retain_metrics = evaluate(unlearned_model, retain_loader, device)
delete_metrics = evaluate(unlearned_model, delete_loader, device)

print("Retain:", retain_metrics)
print("Delete:", delete_metrics)
print("Unlearning time:", unlearn_time)

Epoch 0: loss = 0.2064
Epoch 1: loss = 0.0781
✔ Early checkpoint saved at epoch 1
Epoch 2: loss = 0.0585
Epoch 3: loss = 0.0480
Epoch 4: loss = 0.0411
✔ Full model saved
Retain: {'accuracy': 0.9912631578947368, 'loss': 0.028162270316887568}
Delete: {'accuracy': 0.9796666666666667, 'loss': 0.07448671747771825}
Unlearning time: 37.28353404998779


In [14]:
results = {
    "experiment_info": {
        "timestamp": datetime.now().isoformat(),
        "dataset": "MNIST",
        "model": "SimpleCNN",
        "deletion_ratio": 0.05,
        "device": device,
    },

    "first_epoch_reversal": {
        "method": "first_epoch_reversal",
        "hyperparameters": {
            "rewind_epoch": train_meta["save_epoch"],
            "total_training_epochs": train_meta["total_epochs"],
            "retrain_epochs": 3,
            "optimizer": "Adam",
            "learning_rate": 1e-3
        },
        "metrics": {
            "deleted_loss": delete_metrics["loss"],
            "retained_loss": retain_metrics["loss"],
            "retained_accuracy": retain_metrics["accuracy"]
        },
        "time_seconds": unlearn_time
    }
}

In [15]:
os.makedirs("results", exist_ok=True)

filename = f"results/unlearning_results_{int(time.time())}.json"

with open(filename, "w") as f:
    json.dump(results, f, indent=4)

print(f"✔ Results saved to {filename}")

✔ Results saved to results/unlearning_results_1768803268.json
